# Data preprocessing for supervised machine learning

## What you'll learn in this course 🧐🧐

The example below implements all the necessary steps to train a machine learning model and make predictions on a dataset.

- Use pandas and the preprocessing module of the scikit-learn library to prepare your data.
- Use scikit-learn to train a supervised machine learning model and assess its performance.

We have a sample HR dataset from a European company. The objective is to predict whether an employee has been promoted, based on information about that employee: nationality, department, age, years of experience and education level.

- We will call "variable to predict", "variable to explain" or "target", noted Y, the column corresponding to "Promoted" in the dataset
- The other columns of the dataset, called "explanatory variables" and denoted X, will be used to try to predict the value of Y


## Import useful modules ⬇️⬇️


In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import warnings

warnings.filterwarnings(
    "ignore", category=DeprecationWarning
)  # to avoid deprecation warnings

## File reading and basic exploration 📰📰


In [16]:
# Import dataset
print("Loading dataset...")
dataset = pd.read_csv("src/Dataset_preprocessing.csv")
print("...Done.")
print()

Loading dataset...
...Done.



In [17]:
# Basic stats
print("Number of rows : {}".format(dataset.shape[0]))
print()

print("Display of dataset: ")
display(dataset.head())
print()

print("Basics statistics: ")
data_desc = dataset.describe(include="all")
display(data_desc)
print()

print("Percentage of missing values: ")
display(100 * dataset.isnull().sum() / dataset.shape[0])

Number of rows : 502

Display of dataset: 


,id,Country,Department,Age,YearsExperience,EducationLevel,Salary,Promoted,useless_col,almost_empty
0,0,Germany,Marketing,NaN,15,Bachelor,70000,No,useless,NaN
1,1,Italy,Sales,25.0,1,Bachelor,32000,No,useless,40.0
2,2,UK,Sales,42.0,18,Master,77500,No,useless,NaN
3,3,Spain,Finance,44.0,26,Master,95000,Yes,useless,20.0
4,4,France,Marketing,44.0,16,Master,80000,No,useless,NaN



Basics statistics: 


,id,Country,Department,Age,YearsExperience,EducationLevel,Salary,Promoted,useless_col,almost_empty
count,502.000000,502,502,452.000000,502.000000,502,5.020000e+02,502,502,2.000000
unique,NaN,5,5,NaN,NaN,3,NaN,2,1,NaN
top,NaN,France,Engineering,NaN,NaN,Bachelor,NaN,Yes,useless,NaN
freq,NaN,141,166,NaN,NaN,244,NaN,254,502,NaN
mean,250.500000,NaN,NaN,39.075221,16.239044,NaN,2.068374e+06,NaN,NaN,30.000000
std,145.059183,NaN,NaN,9.885312,9.693642,NaN,4.462878e+07,NaN,NaN,14.142136
min,0.000000,NaN,NaN,-10.000000,0.000000,NaN,2.500000e+04,NaN,NaN,20.000000
25%,125.250000,NaN,NaN,32.000000,9.000000,NaN,5.862500e+04,NaN,NaN,25.000000
50%,250.500000,NaN,NaN,39.000000,16.000000,NaN,7.700000e+04,NaN,NaN,30.000000
75%,375.750000,NaN,NaN,45.000000,23.000000,NaN,9.400000e+04,NaN,NaN,35.000000



Percentage of missing values: 


id                  0.000000
Country             0.000000
Department          0.000000
Age                 9.960159
YearsExperience     0.000000
EducationLevel      0.000000
Salary              0.000000
Promoted            0.000000
useless_col         0.000000
almost_empty       99.601594
dtype: float64

In [18]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 502 entries, 0 to 501
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               502 non-null    int64  
 1   Country          502 non-null    object 
 2   Department       502 non-null    object 
 3   Age              452 non-null    float64
 4   YearsExperience  502 non-null    int64  
 5   EducationLevel   502 non-null    object 
 6   Salary           502 non-null    int64  
 7   Promoted         502 non-null    object 
 8   useless_col      502 non-null    object 
 9   almost_empty     2 non-null      float64
dtypes: float64(2), int64(3), object(5)
memory usage: 39.3+ KB


The exploration of the above data makes it possible to know which pre-processing steps will be necessary:

**1. Preprocessing to be planned with pandas**

**Unnecessary columns for prediction, to be thrown away** :

- _id_ is an identifier, it should never be used for prediction (this column contains no information)
- _useless_col_ will also be useless, because it always contains the same value

**Columns with too many missing values, to be discarded** : _almost_empty_

**Lines containing outliers, discarded** :

- Lines for which _Age_ is negative
- Lines for which _Salary_ is more than 2 standard deviations away (std) (this is not a rule to be applied in general, but here we notice that it allows to discard the value of 1Billion which seems aberrant)

**Target variable/target (Y) that we will try to predict, to separate from the others** : Promoted

**------------**

**2. Preprocessings to be planned with scikit-learn\*\***.

**Explanatory variables (X)**
We need to identify which columns contain categorical variables and which columns contain numerical variables, as they will be treated differently.

- Categorical variables : Country, Department, EducationLevel
- Numerical variables : Age, YearsExperience, Salary

In this dataset, we have both types of variables. It will thus be necessary to plan to create a numeric_transformer (which will call the StandardScaler class) and a categorical_transformer (which will call the OneHotEncoder class). On the other hand, as we observe missing values in the starting dataset, we will have to plan to call the SimpleImputer class to handle the missing values.

**Target variable Y**
Finally, here it should be noted that the variable Y to be predicted is categorical: it will thus be necessary to provide an encoding with the LabelEncoder class.


## Preprocessing - Pandas 🐼🐼


In [19]:
# Drop useless columns / columns with too many missing values
useless_cols = ["id", "useless_col", "almost_empty"]

print("Dropping useless columns...")
dataset = dataset.drop(
    useless_cols, axis=1
)  # axis = 1 indicates that we are dropping along the column axis
# never hesitate to look at a function's documentation using the command name_of_the_function?
print("...Done.")
print(dataset.head())

Dropping useless columns...
...Done.
   Country Department   Age  YearsExperience EducationLevel  Salary Promoted
0  Germany  Marketing   NaN               15       Bachelor   70000       No
1    Italy      Sales  25.0                1       Bachelor   32000       No
2       UK      Sales  42.0               18         Master   77500       No
3    Spain    Finance  44.0               26         Master   95000      Yes
4   France  Marketing  44.0               16         Master   80000       No


In [21]:
# Drop lines containing outliers (using masks)

print("Dropping outliers in Age...")
to_keep = (dataset["Age"] > 0) | (
    dataset["Age"].isnull()
)  # We want keeping positives values or missings
dataset_clean = dataset.loc[to_keep, :]
print("Done. Number of lines remaining : ", dataset.shape[0])
print()

print("Dropping outliers in Salary...")
to_keep = dataset_clean["Salary"] < dataset_clean["Salary"].mean() + 2 * dataset_clean["Salary"].std()
dataset_clean = dataset_clean.loc[to_keep, :]
print("Done. Number of lines remaining : ", dataset.shape[0])
print()

dataset_clean.head()

Dropping outliers in Age...
Done. Number of lines remaining :  502

Dropping outliers in Salary...
Done. Number of lines remaining :  502



,Country,Department,Age,YearsExperience,EducationLevel,Salary,Promoted
0,Germany,Marketing,NaN,15,Bachelor,70000,No
1,Italy,Sales,25.0,1,Bachelor,32000,No
2,UK,Sales,42.0,18,Master,77500,No
3,Spain,Finance,44.0,26,Master,95000,Yes
4,France,Marketing,44.0,16,Master,80000,No


In [7]:
to_keep

0       True
1       True
2       True
3       True
4       True
       ...  
496     True
497     True
498     True
499     True
501    False
Name: Salary, Length: 501, dtype: bool

In [22]:
dataset_clean.head(0)

,Country,Department,Age,YearsExperience,EducationLevel,Salary,Promoted


In [23]:
# Separate target variable Y from features X
target_name = "Promoted"

print("Separating labels from features...")
Y = dataset_clean.loc[:, target_name]
X = dataset_clean.drop(target_name, axis=1)  # All columns are kept, except the target
print("...Done.")
print(Y.head())
print()
print(X.head())
print()

Separating labels from features...
...Done.
0     No
1     No
2     No
3    Yes
4     No
Name: Promoted, dtype: object

   Country Department   Age  YearsExperience EducationLevel  Salary
0  Germany  Marketing   NaN               15       Bachelor   70000
1    Italy      Sales  25.0                1       Bachelor   32000
2       UK      Sales  42.0               18         Master   77500
3    Spain    Finance  44.0               26         Master   95000
4   France  Marketing  44.0               16         Master   80000



In [24]:
Y

0       No
1       No
2       No
3      Yes
4       No
      ... 
495     No
496    Yes
497     No
498     No
499    Yes
Name: Promoted, Length: 500, dtype: object

In [25]:
X

,Country,Department,Age,YearsExperience,EducationLevel,Salary
0,Germany,Marketing,NaN,15,Bachelor,70000
1,Italy,Sales,25.0,1,Bachelor,32000
2,UK,Sales,42.0,18,Master,77500
3,Spain,Finance,44.0,26,Master,95000
4,France,Marketing,44.0,16,Master,80000
...,...,...,...,...,...,...
495,Germany,Engineering,39.0,16,Master,76500
496,Spain,HR,61.0,35,Master,110000
497,France,Engineering,42.0,18,Bachelor,65000
498,Italy,HR,40.0,15,Master,61500


## Preprocessing - Scikit-Learn 🔬🔬

### Train-test splitting

Before doing any preprocessing, let's put a little part (10-20%) of the dataset aside. This data will be refered to as the _test set_ and won't be used to train the model. Later in this notebook, we'll use the biggest part (80-90%) of the data (the _train set_) to train our model, and then the test set will allow us to estimate what are the performances we could expect on new data that has never been seen by the model.

<Note type="tip">

If it's not obvious to you why this train/test splitting is important, don't worry, we'll go deeper into this topic at the end of the module 🤓 for now, you can just blindly apply this simple rule: always make a train/test splitting before you make any transformation on your data using scikit-learn !

</Note>


In [26]:
# First : always divide dataset into train set & test set !!
print("Dividing into train and test sets...")
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=0)
# test_size indicates the proportion of rows from X and Y that will go into the test dataset while
# maintaining the correspondance between the rows from X and Y

# random_state is an argument that can be found in all functions that have a pseudo-random behaviour
# if random_state is not stated the function will derive a different random result everytime the cell
# runs, if random_state is given a value the results will be the same everytime the cell runs while
# each different value of radom_state will derive a specific result
print("...Done.")
print()

Dividing into train and test sets...
...Done.



### Preprocessings : imputation of missing values, standardizing and one-hot encoding

Let's apply the remaining transformations required to prepare the data for model training.

There's one main difficulty: we want to apply different transformations to the features, depending on their types. Numeric columns will be standardized whereas categorical columns will be one-hot-encoded. As regards missing values imputation, it can apply to every column, but different rules may be used. For example, we might want to replace missing values by the median in numeric features, and by the variable's mode in categorical features. You can imagine that, with many different columns and many different types, this can lead to a lot of code with many exceptions 🤯

Fortunately, scikit-learn's `Pipeline` and `ColumnTransformer` classes will help us "tidy" our code and make it as readable as possible while allowing to choose different steps for different columns.

<Note type="note" title="Pipeline vs. ColumnTransformer">

- The `Pipeline` class allows to describe the successive steps that we want to apply to a feature. For example: "1. replace missing values by the median 2. standardize the variable". Once this pipeline has been declared, it's possible to apply it to every column that is concerned.
- The `ColumnTransformer` class allows to apply a given `Pipeline` to a whole set of columns at once

</Note>

<Note type="warning" title="Don't panic!">

This may seem a bit complicated to you at first, but you'll use these classes every single day from now on, and you'll see that it actually always behaves the same. In the end, it's likely that you'll know these lines of code by heart, repeating and practicing are key 😌 But in the beginning, you can settle for copy/pasting these lines whenever you need to make some preprocessings !

</Note>


In [27]:
# Create pipeline for numeric features
numeric_features = ["Age", "YearsExperience", "Salary"]  # Names of numeric columns in X_train/X_test
numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),  # missing values will be replaced by columns' median
        ("scaler", StandardScaler()),
    ]
)

In [28]:
# Create pipeline for categorical features
categorical_features = ["Country", "Department", "EducationLevel"]  # Names of categorical columns in X_train/X_test
categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent"),
        ),  # missing values will be replaced by most frequent value
        (
            "encoder",
            OneHotEncoder(drop="first"),
        ),  # first column will be dropped to avoid creating correlations between features
    ]
)

In [29]:
# Use ColumnTransformer to make a preprocessor object that describes all the treatments to be done
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

In [30]:
# Preprocessings on train set
print("Performing preprocessings on train set...")
print(X_train.head())
X_train = preprocessor.fit_transform(X_train)
print("...Done.")
print(
    X_train[0:5]
)  # MUST use this syntax because X_train is a numpy array and not a pandas DataFrame anymore
print()

# Preprocessings on test set
print("Performing preprocessings on test set...")
print(X_test.head())
X_test = preprocessor.transform(
    X_test
)  # Don't fit again !! The test set is used for validating decisions
# we made based on the training set, therefore we can only apply transformations that were parametered using the training set.
# Otherwise this creates what is called a leak from the test set which will introduce a bias in all your results.
print("...Done.")
print(
    X_test[0:5, :]
)  # MUST use this syntax because X_test is a numpy array and not a pandas DataFrame anymore
print()

Performing preprocessings on train set...
    Country   Department   Age  YearsExperience EducationLevel  Salary
107      UK        Sales  34.0               13       Bachelor   63000
336      UK           HR  62.0               35         Master  122500
71   France      Finance  32.0                6       Bachelor   58500
474   Spain  Engineering  50.0               30       Bachelor  106500
6    France           HR  46.0               27            PhD  107000
...Done.
[[-0.57763389 -0.35263514 -0.59567858  0.          0.          0.
   1.          0.          0.          0.          1.          0.
   0.        ]
 [ 2.46683651  1.91246649  1.97101829  0.          0.          0.
   1.          0.          1.          0.          0.          1.
   0.        ]
 [-0.79509606 -1.0733493  -0.78979851  0.          0.          0.
   0.          1.          0.          0.          0.          0.
   0.        ]
 [ 1.16206348  1.39767066  1.28081409  0.          0.          1.
   0.          0

<Note type="warning" title="Type conversion">

Scikit-learn's ColumnTransformer converts pandas DataFrames into numpy arrays. You can check that by displaying the type of the object returned by the fit_transform() method. The cell below won't execute because all the methods from pandas won't be available anymore once the conversion has been made. To display the first lines of your dataset, you'll have to use the numpy syntax as shown below.

</Note>


In [31]:
# This will create an AttributeError because X_train is not a pandas DataFrame anymore !
X_train.head()

AttributeError: 'numpy.ndarray' object has no attribute 'head'

In [32]:
# Actually, X_train is a numpy array
type(X_train)

numpy.ndarray

In [33]:
# Numpy syntax to display 5 first lines
X_train[0:5]

array([[-0.57763389, -0.35263514, -0.59567858,  0.        ,  0.        ,
         0.        ,  1.        ,  0.        ,  0.        ,  0.        ,
         1.        ,  0.        ,  0.        ],
       [ 2.46683651,  1.91246649,  1.97101829,  0.        ,  0.        ,
         0.        ,  1.        ,  0.        ,  1.        ,  0.        ,
         0.        ,  1.        ,  0.        ],
       [-0.79509606, -1.0733493 , -0.78979851,  0.        ,  0.        ,
         0.        ,  0.        ,  1.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ],
       [ 1.16206348,  1.39767066,  1.28081409,  0.        ,  0.        ,
         1.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ],
       [ 0.72713914,  1.08879317,  1.30238297,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  1.        ,  0.        ,
         0.        ,  0.        ,  1.        ]])

In [34]:
# Encode target variable Y
labelencoder = LabelEncoder()

print("Encoding labels on train set...")
print(Y_train.head())
print()
Y_train = labelencoder.fit_transform(Y_train)
print("...Done.")
print(Y_train[0:5])
print()

print("Encoding labels on test set...")
print(Y_test.head())
print()
Y_test = labelencoder.transform(Y_test)  # Don't fit again !!
print("...Done.")
print(Y_test[0:5])
print()

Encoding labels on train set...
107     No
336    Yes
71     Yes
474    Yes
6      Yes
Name: Promoted, dtype: object

...Done.
[0 1 1 1 1]

Encoding labels on test set...
90      No
254    Yes
283     No
445     No
461    Yes
Name: Promoted, dtype: object

...Done.
[0 1 0 0 1]



### Model training 🏃


In [35]:
# Train model
model = LogisticRegression()

print("Training model...")
model.fit(X_train, Y_train)  # Training is always done on train set !!
print("...Done.")

Training model...
...Done.


### Predictions 🔮


In [36]:
# Predictions on training set
print("Predictions on training set...")
Y_train_pred = model.predict(X_train)
print("...Done.")
print(Y_train_pred[0:5])
print()

Predictions on training set...
...Done.
[0 1 0 1 1]



In [37]:
# Predictions on test set
print("Predictions on test set...")
Y_test_pred = model.predict(X_test)
print("...Done.")
print(Y_test_pred[0:5])
print()

Predictions on test set...
...Done.
[0 1 0 0 1]



### Performance Evaluation 💯

To assess the performances of your model, you can load the function corresponding to the metric you're interested in from the [sklearn.metrics](https://scikit-learn.org/stable/modules/classes.html#module-sklearn.metrics) module. There exist a lot of scoring methods, you'll get familiar with some of them during the next lectures. In this demo, we use the accuracy score, which is the most widespread score for classification problems.


In [38]:
# Print scores
print(
    "Accuracy on training set : ", accuracy_score(Y_train, Y_train_pred)
)  # Always pass true label first, and predictions in second position
print("Accuracy on test set : ", accuracy_score(Y_test, Y_test_pred))

Accuracy on training set :  0.655
Accuracy on test set :  0.66


### Alternative way of computing the accuracy score

In scikit-learn, every class implementing a machine learning model has a `score()` method that can be used to assess the score without explicitely computing the model's predictions beforehand.

<Note type="warning" title="Always check the metric used !">

The `score()` method will return a basic score that is consistent with the type of problem (in this demo it's the accuracy). To be sure what metric is used, you can refer to the [model's documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html#sklearn.linear_model.LogisticRegression.score).

</Note>


In [ ]:
# Print scores
print(
    "Accuracy on training set : ", model.score(X_train, Y_train)
)  # Here, the features must be passed first, and then the true label
print("Accuracy on test set : ", model.score(X_test, Y_test))

## Resources 📚📚

- Introduction Pratique à Python - Antoine Krajnc-Rosenthal & Anais Armandy
- Sklearn - [Documentation officielle de scikit-learn](https://scikit-learn.org/stable/)


In [40]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import warnings

warnings.filterwarnings(
    "ignore", category=DeprecationWarning
)  # to avoid deprecation warnings


# Import dataset
print("Loading dataset...")
dataset = pd.read_csv("src/Dataset_preprocessing.csv")
print("...Done.")
print()

# Drop useless columns / columns with too many missing values
useless_cols = ["id", "useless_col", "almost_empty"]

print("Dropping useless columns...")
dataset = dataset.drop(
    useless_cols, axis=1
) 

print("Dropping outliers in Age...")
to_keep = (dataset["Age"] > 0) | (
    dataset["Age"].isnull()
)  # We want keeping positives values or missings
dataset_clean = dataset.loc[to_keep, :]
print("Done. Number of lines remaining : ", dataset.shape[0])
print()

print("Dropping outliers in Salary...")
to_keep = dataset_clean["Salary"] < dataset_clean["Salary"].mean() + 2 * dataset_clean["Salary"].std()
dataset_clean = dataset_clean.loc[to_keep, :]
print("Done. Number of lines remaining : ", dataset.shape[0])
print()


# Separate target variable Y from features X
target_name = "Promoted"

print("Separating labels from features...")
Y = dataset_clean.loc[:, target_name]
X = dataset_clean.drop(target_name, axis=1)  # All columns are kept, except the target
print("...Done.")
print(Y.head())
print()
print(X.head())
print()



Loading dataset...
...Done.

Dropping useless columns...
Dropping outliers in Age...
Done. Number of lines remaining :  502

Dropping outliers in Salary...
Done. Number of lines remaining :  502

Separating labels from features...
...Done.
0     No
1     No
2     No
3    Yes
4     No
Name: Promoted, dtype: object

   Country Department   Age  YearsExperience EducationLevel  Salary
0  Germany  Marketing   NaN               15       Bachelor   70000
1    Italy      Sales  25.0                1       Bachelor   32000
2       UK      Sales  42.0               18         Master   77500
3    Spain    Finance  44.0               26         Master   95000
4   France  Marketing  44.0               16         Master   80000



In [42]:
from preprocess_data import preprocess
X,Y = preprocess("src/Dataset_preprocessing.csv")

Loading dataset...
...Done.

Dropping useless columns...
Dropping outliers in Age...
Done. Number of lines remaining :  502

Dropping outliers in Salary...
Done. Number of lines remaining :  502

Separating labels from features...
...Done.
0     No
1     No
2     No
3    Yes
4     No
Name: Promoted, dtype: object

   Country Department   Age  YearsExperience EducationLevel  Salary
0  Germany  Marketing   NaN               15       Bachelor   70000
1    Italy      Sales  25.0                1       Bachelor   32000
2       UK      Sales  42.0               18         Master   77500
3    Spain    Finance  44.0               26         Master   95000
4   France  Marketing  44.0               16         Master   80000



In [43]:
X

,Country,Department,Age,YearsExperience,EducationLevel,Salary
0,Germany,Marketing,NaN,15,Bachelor,70000
1,Italy,Sales,25.0,1,Bachelor,32000
2,UK,Sales,42.0,18,Master,77500
3,Spain,Finance,44.0,26,Master,95000
4,France,Marketing,44.0,16,Master,80000
...,...,...,...,...,...,...
495,Germany,Engineering,39.0,16,Master,76500
496,Spain,HR,61.0,35,Master,110000
497,France,Engineering,42.0,18,Bachelor,65000
498,Italy,HR,40.0,15,Master,61500


In [44]:
Y

0       No
1       No
2       No
3      Yes
4       No
      ... 
495     No
496    Yes
497     No
498     No
499    Yes
Name: Promoted, Length: 500, dtype: object

In [41]:
# First : always divide dataset into train set & test set !!
print("Dividing into train and test sets...")
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=0)
print("...Done.")

# Create pipeline for numeric features
numeric_features = ["Age", "YearsExperience", "Salary"]  # Names of numeric columns in X_train/X_test
numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),  # missing values will be replaced by columns' median
        ("scaler", StandardScaler()),
    ]
)


# Create pipeline for categorical features
categorical_features = ["Country", "Department", "EducationLevel"]  # Names of categorical columns in X_train/X_test
categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent"),
        ),  # missing values will be replaced by most frequent value
        (
            "encoder",
            OneHotEncoder(drop="first"),
        ),  # first column will be dropped to avoid creating correlations between features
    ]
)

# Use ColumnTransformer to make a preprocessor object that describes all the treatments to be done
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# Preprocessings on train set
print("Performing preprocessings on train set...")
print(X_train.head())
X_train = preprocessor.fit_transform(X_train)
print("...Done.")
print(
    X_train[0:5]
)  # MUST use this syntax because X_train is a numpy array and not a pandas DataFrame anymore
print()

# Preprocessings on test set
print("Performing preprocessings on test set...")
print(X_test.head())
X_test = preprocessor.transform(
    X_test
)  # Don't fit again !! The test set is used for validating decisions
# we made based on the training set, therefore we can only apply transformations that were parametered using the training set.
# Otherwise this creates what is called a leak from the test set which will introduce a bias in all your results.
print("...Done.")
print(
    X_test[0:5, :]
)  # MUST use this syntax because X_test is a numpy array and not a pandas DataFrame anymore
print()

Dividing into train and test sets...
...Done.
Performing preprocessings on train set...
    Country   Department   Age  YearsExperience EducationLevel  Salary
107      UK        Sales  34.0               13       Bachelor   63000
336      UK           HR  62.0               35         Master  122500
71   France      Finance  32.0                6       Bachelor   58500
474   Spain  Engineering  50.0               30       Bachelor  106500
6    France           HR  46.0               27            PhD  107000
...Done.
[[-0.57763389 -0.35263514 -0.59567858  0.          0.          0.
   1.          0.          0.          0.          1.          0.
   0.        ]
 [ 2.46683651  1.91246649  1.97101829  0.          0.          0.
   1.          0.          1.          0.          0.          1.
   0.        ]
 [-0.79509606 -1.0733493  -0.78979851  0.          0.          0.
   0.          1.          0.          0.          0.          0.
   0.        ]
 [ 1.16206348  1.39767066  1.2808140